# Grey-pixel segmentation (Gehler-Shi 568, PyTorch)

Binary segmentation: **raw linear 12-bit PNG** (RGB, $\gamma=1$, counts $\approx 0\ldots4095$) $\rightarrow$ probability map of achromatic (grey) pixels.

**Ground truth** `preprocessed_canon1d/masks`: masks built from **white-balanced** images using **dataset ground-truth illuminations** (see `preprocess_canon1d.py`). The network learns to predict that mask from **raw** inputs.

**Evaluation**: metrics are computed only on **scene pixels** (color-checker regions excluded via `cc_masks`), matching how masks were generated.


In [ ]:
# %% 1. Imports, config, optional preprocessing
from __future__ import annotations

import inspect
import os
import random
import subprocess
import sys
import warnings
from pathlib import Path

# ── Suppress noisy non-error output ──────────────────────────────────────────
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no huggingface download bars
os.environ["TQDM_DISABLE"] = "1"                    # disable tqdm globally

import matplotlib
matplotlib.use("Agg")   # non-interactive backend: no display, no widget blobs
import matplotlib.pyplot as plt

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# --- config ---
RUN_PREPROCESS = True  # run preprocess_canon1d.py from repo root before training

# Resolve repo root (directory that contains preprocess_canon1d.py)
PROJECT_ROOT = Path.cwd()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / "preprocess_canon1d.py").is_file():
        PROJECT_ROOT = p
        break

IMG_DIR = Path(
    r"C:\Users\bauti\Desktop\IMLEX\UJM\Deep Learning\Project - Grey Color Detection\png_canon1d\cs\chroma\data\canon_dataset\568_dataset\png"
)
OUT_DIR = PROJECT_ROOT / "preprocessed_canon1d"
MASK_DIR = OUT_DIR / "masks"
CC_DIR = OUT_DIR / "cc_masks"
ARTIFACT_DIR = PROJECT_ROOT / "training" / "artifacts"
CHECKPOINT_PATH = ARTIFACT_DIR / "best.pt"

# training hyperparameters
SEED = 42
PATCH_SIZE = 512
BATCH_SIZE = 8  # raise if CUDA OOM; lower if needed
NUM_EPOCHS = 40
LR = 3e-4
WEIGHT_DECAY = 1e-2
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # test = remainder
EXPECTED_DATASET_SIZE = 568  # full Gehler-Shi 568 dataset
NUM_WORKERS = 0  # Windows + nbconvert: keep 0 (multiprocessing often breaks workers>0)

DIVISIBLE_BY = 32  # U-Net / ResNet stride

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
print("PROJECT_ROOT :", PROJECT_ROOT.resolve())
print("IMG_DIR      :", IMG_DIR)
print("device       :", device)

if RUN_PREPROCESS:
    script = PROJECT_ROOT / "preprocess_canon1d.py"
    if not script.is_file():
        raise FileNotFoundError(script)
    subprocess.run([sys.executable, str(script)], cwd=str(PROJECT_ROOT), check=True)
else:
    print("Skipping preprocess (set RUN_PREPROCESS = True to run).")


## 2. Data: pair raw PNGs, masks, and color-checker exclusions

In [ ]:
def list_paired_stems(img_dir: Path, mask_dir: Path, cc_dir: Path) -> list[str]:
    stems = []
    for p in sorted(img_dir.glob("*.png")):
        s = p.stem
        if (mask_dir / f"{s}_grey_mask.png").is_file() and (cc_dir / f"{s}_cc_mask.png").is_file():
            stems.append(s)
    return stems


def read_raw_png_rgb(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if bgr is None:
        raise FileNotFoundError(path)
    if bgr.ndim == 2:
        bgr = np.stack([bgr, bgr, bgr], axis=-1)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.uint16)


def read_mask_binary(path: Path) -> np.ndarray:
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if g is None:
        raise FileNotFoundError(path)
    return (g > 127).astype(np.float32)


def read_cc_mask(path: Path) -> np.ndarray:
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if g is None:
        raise FileNotFoundError(path)
    return (g > 127).astype(np.float32)


def pad_to_divisible_hw(img: np.ndarray, mask: np.ndarray, cc: np.ndarray, div: int = DIVISIBLE_BY):
    h, w = img.shape[:2]
    nh = (h + div - 1) // div * div
    nw = (w + div - 1) // div * div
    ph, pw = nh - h, nw - w
    if ph or pw:
        img  = np.pad(img,  ((0, ph), (0, pw), (0, 0)), mode="reflect")
        mask = np.pad(mask, ((0, ph), (0, pw)),          mode="reflect")
        cc   = np.pad(cc,   ((0, ph), (0, pw)),          mode="reflect")
    return img, mask, cc, (h, w)


def random_crop_triplet(img_f, mask, cc, patch, rng):
    h, w = img_f.shape[:2]
    if h < patch or w < patch:
        ph = max(0, patch - h); pw = max(0, patch - w)
        img_f = np.pad(img_f, ((0, ph), (0, pw), (0, 0)), mode="reflect")
        mask  = np.pad(mask,  ((0, ph), (0, pw)),          mode="reflect")
        cc    = np.pad(cc,    ((0, ph), (0, pw)),          mode="reflect")
        h, w  = img_f.shape[:2]
    top  = rng.randint(0, h - patch)
    left = rng.randint(0, w - patch)
    sl   = np.s_[top:top + patch, left:left + patch]
    return img_f[sl], mask[sl], cc[sl]


class GreySegDataset(Dataset):
    def __init__(self, stems, img_dir, mask_dir, cc_dir, patch, train, seed):
        self.stems    = stems
        self.img_dir  = img_dir
        self.mask_dir = mask_dir
        self.cc_dir   = cc_dir
        self.patch    = patch
        self.train    = train
        self._rng     = random.Random(seed)

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        s     = self.stems[idx]
        raw   = read_raw_png_rgb(self.img_dir / f"{s}.png")
        img_f = (raw.astype(np.float32) / 4095.0).clip(0.0, 1.0)
        mask  = read_mask_binary(self.mask_dir / f"{s}_grey_mask.png")
        cc    = read_cc_mask(self.cc_dir    / f"{s}_cc_mask.png")

        if self.train and self.patch is not None:
            img_f, mask, cc = random_crop_triplet(img_f, mask, cc, self.patch, self._rng)
            if self._rng.random() < 0.5:
                img_f = np.ascontiguousarray(img_f[:, ::-1])
                mask  = np.ascontiguousarray(mask[:, ::-1])
                cc    = np.ascontiguousarray(cc[:, ::-1])
        else:
            img_f, mask, cc, _ = pad_to_divisible_hw(img_f, mask, cc)

        valid = 1.0 - cc
        x = torch.from_numpy(img_f).permute(2, 0, 1).float()
        y = torch.from_numpy(mask).unsqueeze(0).float()
        v = torch.from_numpy(valid).unsqueeze(0).float()
        return x, y, v, s


stems = list_paired_stems(IMG_DIR, MASK_DIR, CC_DIR)
if not stems:
    raise RuntimeError(
        f"No paired samples found. Ensure preprocessing ran and paths exist:\n  {MASK_DIR}\n  {CC_DIR}"
    )

n_png = len(list(IMG_DIR.glob("*.png")))
print(f"Dataset check: {n_png} PNGs in IMG_DIR, {len(stems)} fully paired (image + mask + cc_mask)")
if len(stems) != EXPECTED_DATASET_SIZE:
    raise RuntimeError(
        f"Expected {EXPECTED_DATASET_SIZE} paired samples, found {len(stems)}. "
        f"Re-run preprocess_canon1d.py or fix IMG_DIR / preprocessed_canon1d paths."
    )
print(f"OK — using full {EXPECTED_DATASET_SIZE}-image dataset.")

rng_split = random.Random(SEED)
order = stems[:]
rng_split.shuffle(order)
n = len(order)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)
train_stems = order[:n_train]
val_stems   = order[n_train:n_train + n_val]
test_stems  = order[n_train + n_val:]

print(f"Total paired : {n}")
print(f"Train / val / test : {len(train_stems)} / {len(val_stems)} / {len(test_stems)}")

train_ds = GreySegDataset(train_stems, IMG_DIR, MASK_DIR, CC_DIR, PATCH_SIZE, train=True,  seed=SEED)
val_ds   = GreySegDataset(val_stems,   IMG_DIR, MASK_DIR, CC_DIR, patch=None,  train=False, seed=SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   batch_size=1,          shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


## 3. Model: U-Net + ResNet-18 encoder (segmentation_models_pytorch)

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,
)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


## 4. Loss, metrics, training / validation

In [ ]:
def masked_bce_logits(logits, target, valid):
    loss  = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    w     = valid.clamp(0, 1)
    return (loss * w).sum() / w.sum().clamp_min(1.0)


def masked_dice_loss(logits, target, valid, eps=1e-6):
    prob  = torch.sigmoid(logits)
    w     = valid.clamp(0, 1)
    inter = (prob * target * w).sum()
    denom = ((prob + target) * w).sum().clamp_min(eps)
    return 1.0 - (2 * inter + eps) / (denom + eps)


def total_loss(logits, target, valid, bce_w=0.5, dice_w=0.5):
    return bce_w * masked_bce_logits(logits, target, valid) + dice_w * masked_dice_loss(logits, target, valid)


@torch.no_grad()
def binary_metrics(logits, target, valid, thresh=0.5):
    prob  = torch.sigmoid(logits)
    pred  = (prob > thresh).float()
    w     = valid.clamp(0, 1)
    tp = (pred * target * w).sum()
    fp = (pred * (1.0 - target) * w).sum()
    fn = ((1.0 - pred) * target * w).sum()
    tn = ((1.0 - pred) * (1.0 - target) * w).sum()
    iou  = tp / (tp + fp + fn).clamp_min(1e-6)
    prec = tp / (tp + fp).clamp_min(1e-6)
    rec  = tp / (tp + fn).clamp_min(1e-6)
    f1   = 2 * prec * rec / (prec + rec).clamp_min(1e-6)
    acc  = (tp + tn) / w.sum().clamp_min(1e-6)
    dice = 2 * tp / (2 * tp + fp + fn).clamp_min(1e-6)
    return {"iou": iou, "dice": dice, "precision": prec, "recall": rec, "f1": f1, "accuracy": acc,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def train_one_epoch(model, loader, optim, scaler=None):
    model.train()
    losses = []
    for x, y, v, _ in loader:
        x, y, v = x.to(device), y.to(device), v.to(device)
        optim.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.autocast(device_type="cuda"):
                logits = model(x)
                loss   = total_loss(logits, y, v)
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
        else:
            logits = model(x)
            loss   = total_loss(logits, y, v)
            loss.backward()
            optim.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def evaluate_loader(model, loader):
    model.eval()
    tot = {"loss": 0.0, "iou": 0.0, "dice": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0, "accuracy": 0.0}
    n_batches = 0
    for x, y, v, _ in loader:
        x, y, v  = x.to(device), y.to(device), v.to(device)
        logits   = model(x)
        loss     = total_loss(logits, y, v)
        m        = binary_metrics(logits, y, v)
        tot["loss"] += loss.item()
        for k in ("iou", "dice", "precision", "recall", "f1", "accuracy"):
            tot[k] += m[k].item()
        n_batches += 1
    for k in tot:
        tot[k] /= max(n_batches, 1)
    return tot


ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
optim     = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=max(NUM_EPOCHS, 1))
scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

history  = {k: [] for k in ["train_loss", "val_loss", "val_iou", "val_dice", "val_f1", "lr"]}
best_iou = -1.0
best_epoch = -1

print(f"{'Epoch':>5}  {'TrainLoss':>9}  {'ValLoss':>8}  {'ValIoU':>7}  {'ValDice':>8}  {'ValF1':>7}  {'LR':>9}")
print("-" * 65)
for epoch in range(1, NUM_EPOCHS + 1):
    tr      = train_one_epoch(model, train_loader, optim, scaler)
    va      = evaluate_loader(model, val_loader)
    scheduler.step()
    lr_now  = optim.param_groups[0]["lr"]
    history["train_loss"].append(tr)
    history["val_loss"].append(va["loss"])
    history["val_iou"].append(va["iou"])
    history["val_dice"].append(va["dice"])
    history["val_f1"].append(va["f1"])
    history["lr"].append(lr_now)
    marker = "  *" if va["iou"] > best_iou else ""
    print(f"{epoch:>5}  {tr:>9.4f}  {va['loss']:>8.4f}  {va['iou']:>7.4f}  {va['dice']:>8.4f}  {va['f1']:>7.4f}  {lr_now:>9.2e}{marker}")
    if va["iou"] > best_iou:
        best_iou   = va["iou"]
        best_epoch = epoch
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_iou": best_iou,
                    "stems": {"train": train_stems, "val": val_stems, "test": test_stems}},
                   CHECKPOINT_PATH)

print(f"\nBest val IoU = {best_iou:.4f}  at epoch {best_epoch}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")


### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history["val_iou"],  label="IoU")
axes[1].plot(history["val_dice"], label="Dice")
axes[1].plot(history["val_f1"],   label="F1")
axes[1].set_title("Val metrics"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = ARTIFACT_DIR / "training_curves.png"
plt.savefig(curves_path, dpi=100)
plt.close()
print(f"Training curves saved to: {curves_path}")


## 5. Test (held-out split): aggregate statistics

Metrics use **scene pixels only** (color-checker region excluded).

In [ ]:
@torch.no_grad()
def predict_full_image(model, img_rgb_float: np.ndarray) -> np.ndarray:
    model.eval()
    h0, w0  = img_rgb_float.shape[:2]
    z       = np.zeros((h0, w0), np.float32)
    img, _, _, _ = pad_to_divisible_hw(img_rgb_float, z, z)
    x       = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().to(device)
    prob    = torch.sigmoid(model(x))[0, 0, :h0, :w0].cpu().numpy()
    return prob


def evaluate_split_per_image(model, stems_list):
    rows   = []
    totals = {"tp": 0.0, "fp": 0.0, "fn": 0.0, "tn": 0.0}
    for s in stems_list:
        img_f = (read_raw_png_rgb(IMG_DIR / f"{s}.png").astype(np.float32) / 4095.0).clip(0.0, 1.0)
        mask  = read_mask_binary(MASK_DIR / f"{s}_grey_mask.png")
        cc    = read_cc_mask(CC_DIR / f"{s}_cc_mask.png")
        valid = 1.0 - cc
        pred  = (predict_full_image(model, img_f) > 0.5).astype(np.float32)
        pred[cc > 0.5] = 0  # exclude colorchecker region
        tp = float((pred * mask * valid).sum())
        fp = float((pred * (1 - mask) * valid).sum())
        fn = float(((1 - pred) * mask * valid).sum())
        tn = float(((1 - pred) * (1 - mask) * valid).sum())
        union = tp + fp + fn
        iou  = tp / union if union > 0 else 1.0
        dice = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 1.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 1.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        acc  = (tp + tn) / valid.sum() if valid.sum() > 0 else 1.0
        rows.append({"stem": s, "iou": iou, "dice": dice, "precision": prec, "recall": rec, "f1": f1, "accuracy": acc})
        for k in ("tp", "fp", "fn", "tn"):
            totals[k] += locals()[k]
    return totals, rows


def print_split_summary(rows, totals, name):
    if not rows:
        print(f"{name}: empty split"); return
    keys = ["iou", "dice", "precision", "recall", "f1", "accuracy"]
    print(f"\n{'='*55}")
    print(f"  {name}  (n={len(rows)} images, scene pixels only)")
    print(f"{'='*55}")
    print(f"  {'metric':12s}   {'mean':>7}   {'std':>7}   {'min':>7}   {'max':>7}")
    print(f"  {'-'*51}")
    for k in keys:
        vals = np.array([r[k] for r in rows])
        print(f"  {k:12s}   {vals.mean():7.4f}   {vals.std():7.4f}   {vals.min():7.4f}   {vals.max():7.4f}")
    tp, fp, fn, tn = totals["tp"], totals["fp"], totals["fn"], totals["tn"]
    union      = tp + fp + fn
    micro_iou  = tp / union if union > 0 else 1.0
    micro_dice = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 1.0
    print(f"\n  Global pixel counts  TP={tp:.0f}  FP={fp:.0f}  FN={fn:.0f}  TN={tn:.0f}")
    print(f"  Micro IoU={micro_iou:.4f}   Micro Dice={micro_dice:.4f}")

    print(f"\n  Per-image breakdown:")
    print(f"  {'image':22s}  {'IoU':>6}  {'Dice':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}")
    print(f"  {'-'*58}")
    for r in sorted(rows, key=lambda x: x["iou"], reverse=True):
        print(f"  {r['stem']:22s}  {r['iou']:6.4f}  {r['dice']:6.4f}  {r['precision']:6.4f}  {r['recall']:6.4f}  {r['f1']:6.4f}")


_load_kw = {"map_location": device}
if "weights_only" in inspect.signature(torch.load).parameters:
    _load_kw["weights_only"] = False
ckpt = torch.load(CHECKPOINT_PATH, **_load_kw)
model.load_state_dict(ckpt["model"])
print(f"Loaded checkpoint  epoch={ckpt.get('epoch')}  val_iou={ckpt.get('val_iou'):.4f}")

totals_test, per_image = evaluate_split_per_image(model, test_stems)
print_split_summary(per_image, totals_test, "Held-out test")


## 6. Visual inspection (inference only — no retraining)

**Run only this cell** after training is done. It loads `training/artifacts/best.pt` and compares predicted vs ground-truth masks.

Each row: input (display γ) | predicted mask | GT mask | overlay (G=TP, R=FP, B=FN; color-checker excluded).


In [ ]:
# %% Inference-only visual check — does NOT retrain; loads best.pt
from __future__ import annotations

import inspect
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import segmentation_models_pytorch as smp

# --- user knobs ---
DEMO_STEMS = None   # e.g. ["IMG_0755", "8D5U5560"]; None = sample from test split in checkpoint
SHOW_N = 4
PRED_THRESH = 0.5
SEED = 42

# --- paths (same as training notebook) ---
PROJECT_ROOT = Path.cwd()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / "preprocess_canon1d.py").is_file():
        PROJECT_ROOT = p
        break

IMG_DIR = Path(
    r"C:\Users\bauti\Desktop\IMLEX\UJM\Deep Learning\Project - Grey Color Detection\png_canon1d\cs\chroma\data\canon_dataset\568_dataset\png"
)
OUT_DIR = PROJECT_ROOT / "preprocessed_canon1d"
MASK_DIR = OUT_DIR / "masks"
CC_DIR = OUT_DIR / "cc_masks"
CHECKPOINT_PATH = PROJECT_ROOT / "training" / "artifacts" / "best.pt"
DIVISIBLE_BY = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("checkpoint:", CHECKPOINT_PATH)


def read_raw_png_rgb(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if bgr is None:
        raise FileNotFoundError(path)
    if bgr.ndim == 2:
        bgr = np.stack([bgr, bgr, bgr], axis=-1)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.uint16)


def read_mask_binary(path: Path) -> np.ndarray:
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if g is None:
        raise FileNotFoundError(path)
    return (g > 127).astype(np.float32)


def read_cc_mask(path: Path) -> np.ndarray:
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if g is None:
        raise FileNotFoundError(path)
    return (g > 127).astype(np.float32)


def pad_to_divisible_hw(img, mask, cc, div=DIVISIBLE_BY):
    h, w = img.shape[:2]
    nh = (h + div - 1) // div * div
    nw = (w + div - 1) // div * div
    ph, pw = nh - h, nw - w
    if ph or pw:
        img = np.pad(img, ((0, ph), (0, pw), (0, 0)), mode="reflect")
        mask = np.pad(mask, ((0, ph), (0, pw)), mode="reflect")
        cc = np.pad(cc, ((0, ph), (0, pw)), mode="reflect")
    return img, mask, cc, (h, w)


@torch.no_grad()
def predict_full_image(model, img_rgb_float: np.ndarray) -> np.ndarray:
    model.eval()
    h0, w0 = img_rgb_float.shape[:2]
    z = np.zeros((h0, w0), np.float32)
    img, _, _, _ = pad_to_divisible_hw(img_rgb_float, z, z)
    x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().to(device)
    return torch.sigmoid(model(x))[0, 0, :h0, :w0].cpu().numpy()


def scene_iou(pred_bin, gt, cc):
    valid = cc < 0.5
    p = pred_bin.astype(bool) & valid
    g = gt.astype(bool) & valid
    inter = (p & g).sum()
    union = (p | g).sum()
    return inter / union if union > 0 else 1.0


# --- load trained weights ---
_load_kw = {"map_location": device}
if "weights_only" in inspect.signature(torch.load).parameters:
    _load_kw["weights_only"] = False
ckpt = torch.load(CHECKPOINT_PATH, **_load_kw)

model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None,
).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded epoch={ckpt.get('epoch')}  val_iou={ckpt.get('val_iou', 0):.4f}")

pool = DEMO_STEMS
if not pool:
    pool = ckpt.get("stems", {}).get("test")
if not pool:
    pool = [p.stem for p in sorted(IMG_DIR.glob("*.png"))]
if DEMO_STEMS:
    pool = [s for s in DEMO_STEMS if (IMG_DIR / f"{s}.png").is_file()]
else:
    rng = random.Random(SEED)
    pool = pool[:]
    rng.shuffle(pool)
    pool = pool[:SHOW_N]

print("Showing:", pool)

%matplotlib inline

n = len(pool)
fig, axes = plt.subplots(n, 4, figsize=(16, 3.6 * n))
if n == 1:
    axes = np.expand_dims(axes, 0)

for i, s in enumerate(pool):
    raw = read_raw_png_rgb(IMG_DIR / f"{s}.png")
    img_f = (raw.astype(np.float32) / 4095.0).clip(0.0, 1.0)
    gt = read_mask_binary(MASK_DIR / f"{s}_grey_mask.png")
    cc = read_cc_mask(CC_DIR / f"{s}_cc_mask.png")
    pred = (predict_full_image(model, img_f) > PRED_THRESH).astype(np.float32)
    pred[cc > 0.5] = 0  # exclude colorchecker region
    vis_in = np.power(img_f, 1 / 2.2).clip(0, 1)
    iou = scene_iou(pred, gt, cc)
    scene = cc < 0.5

    axes[i, 0].imshow(vis_in)
    axes[i, 0].set_title(f"{s}\ninput (display γ)")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(pred, vmin=0, vmax=1, cmap="gray")
    axes[i, 1].set_title(f"predicted (t={PRED_THRESH})")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(gt, vmin=0, vmax=1, cmap="gray")
    axes[i, 2].set_title("ground truth")
    axes[i, 2].axis("off")

    p = pred.astype(bool) & scene
    g = gt.astype(bool) & scene
    overlay = np.zeros((*pred.shape, 3), np.float32)
    overlay[..., 1] = p & g
    overlay[..., 0] = p & ~g
    overlay[..., 2] = ~p & g
    axes[i, 3].imshow(0.65 * vis_in + 0.35 * overlay)
    axes[i, 3].set_title(f"errors  IoU={iou:.3f}\nG=TP R=FP B=FN")
    axes[i, 3].axis("off")

    print(f"{s:22s}  scene IoU={iou:.4f}  grey pred={pred[scene].mean():.3f}  grey gt={gt[scene].mean():.3f}")

plt.tight_layout()
plt.show()
